# Fine-Tuning System One Decision Model di Google Colab
## ModernBERT-base (149M) + Semantic Cosine Head + 3-Source Data Mixture (Typed Decisions, BANKING77, CLINC150)

Notebook ini mengimplementasikan resep lengkap **System One Decision Engine** mengikuti pola arsitektur Laya dan data mixture `unsloth_kaggle_dec.py`:

### 1. Masalah yang Diperbaiki dari Percobaan Sebelumnya:
* **Arsitektur Semantic Matcher**: Mengganti linear head acak dengan **Cosine Similarity Proyeksi Laten** antara representasi global State ($s$) dan representasi kontekstual tiap opsi ($o_k$). Model secara alami memanfaatkan pemahaman semantik pra-latih XLM-RoBERTa alih-alih menebak acak.
* **Data Mixture 3 Sumber**: Menggabungkan data latih **`LocalLLaMA/typed-decisions` (100% full)**, **`BANKING77` (2.500 sampel train)**, dan **`CLINC150` (2.500 sampel train)**. Model diajari pola keputusan alur kerja software, perbankan, dan percakapan umum secara bersamaan.
* **Skema Penuh (Full Options)**: ModernBERT ctx 8k muat 77/150 opsi sekaligus (budget 4096 token) — tanpa candidate sampling, tanpa truncation; baris yang meluap di-skip tercatat.
* **Differentiated Learning Rates**: Backbone dilatih dengan lr lembut (`2e-5`), sedangkan Projection Head baru dilatih dengan lr lebih agresif (`5e-4`).
* **Perbaikan Bug Ekstraksi Label**: Kolom `gold` yang bertipe dictionary pada `typed-decisions` diekstrak dengan benar sehingga tidak ada sampel yang terbuang.

## 1. Setup Lingkungan & Deteksi GPU

In [1]:
!pip install -q transformers datasets accelerate huggingface_hub tqdm

import torch
print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU Aktif: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("PERINGATAN: GPU tidak terdeteksi! Ubah runtime ke GPU T4 di Runtime -> Change runtime type.")

PyTorch: 2.11.0+cu130
GPU Aktif: Tesla T4
Total VRAM: 15.64 GB


## 2. Konfigurasi Pelatihan & Hyperparameter

In [11]:
import os

MODEL_ID = "answerdotai/ModernBERT-base"
EPOCHS = 5
BATCH_SIZE = 4            # Baris 4k-token: batch 4 + accum 4 = efektif 16
GRAD_ACCUM = 4
LR_BACKBONE = 2e-5       # Lr lembut untuk pretrained weights
LR_HEAD = 5e-4           # Lr lebih tinggi untuk projection head baru
N_SAMPLES_BANKING = 2500 # Sampel train BANKING77 yang diikutsertakan
N_SAMPLES_CLINC = 2500   # Sampel train CLINC150 yang diikutsertakan
SEED = 3407
OUTPUT_DIR = "xlm-roberta-systemone-decision"

# Opsional: Kredensial Hugging Face Hub untuk auto push (JANGAN commit token!)
HF_REPO = "itsdevice/modernbert-systemone"   # Contoh: "username/modernbert-decision-engine"
try:
    from getpass import getpass
    HF_TOKEN = getpass("HF token (kosongkan bila skip push): ") or ""
except Exception:
    HF_TOKEN = ""

print(f"Model: {MODEL_ID} | Epochs: {EPOCHS} | Batch Size: {BATCH_SIZE}")

Model: FacebookAI/xlm-roberta-base | Epochs: 5 | Batch Size: 16


## 3. Arsitektur `SemanticDecisionModel` (State-Option Cosine Alignment)

Berbeda dari random linear projection, model ini mengekstrak vektor State dari token `<s>` dan vektor tiap Opsi dari token `<mask_k>`, lalu menghitung **Cosine Similarity terukur suhu**.

In [3]:
import math
import torch.nn as nn
import torch.nn.functional as F
from transformers import PreTrainedModel

class SemanticDecisionModel(nn.Module):
    def __init__(self, backbone: PreTrainedModel, hidden_size: int, proj_dim: int = 256):
        super().__init__()
        self.backbone = backbone

        # Proyeksi laten State & Option ke ruang semantik bersama (proj_dim = 256)
        self.state_proj = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, proj_dim),
            nn.GELU(),
            nn.Linear(proj_dim, proj_dim),
        )
        self.opt_proj = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, proj_dim),
            nn.GELU(),
            nn.Linear(proj_dim, proj_dim),
        )

        # Learnable scale (faktor ketajaman kesamaan kosinus) & temperatur
        self.scale = nn.Parameter(torch.tensor(10.0))
        self.log_temp = nn.Parameter(torch.zeros(()))

    @property
    def temperature(self) -> torch.Tensor:
        return self.log_temp.exp().clamp(min=0.05, max=10.0)

    def forward(self, input_ids, attention_mask, mask_positions, mask_valid, labels=None, label_smoothing=0.05):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        H = outputs.last_hidden_state  # (B, L, D)

        B, max_opts = mask_positions.shape
        D = H.size(-1)

        # 1. State Global Vector (diambil dari token <s> di index 0)
        state_vec = self.state_proj(H[:, 0, :])  # (B, proj_dim)
        state_norm = F.normalize(state_vec, p=2, dim=-1)

        # 2. Option Contextual Vectors (diambil dari posisi token <mask_k> tiap opsi)
        idx_exp = mask_positions.unsqueeze(-1).expand(B, max_opts, D)
        mask_reps = torch.gather(H, dim=1, index=idx_exp)  # (B, max_opts, D)
        opt_vec = self.opt_proj(mask_reps)                 # (B, max_opts, proj_dim)
        opt_norm = F.normalize(opt_vec, p=2, dim=-1)

        # 3. Hitung Semantic Cosine Similarity (State vs Tiap Opsi)
        # (B, max_opts, proj_dim) x (B, proj_dim, 1) -> (B, max_opts)
        cos_sim = torch.bmm(opt_norm, state_norm.unsqueeze(-1)).squeeze(-1)
        logits = cos_sim * self.scale
        logits = logits.masked_fill(~mask_valid, -1e4)

        scaled_logits = logits / self.temperature
        probs = F.softmax(scaled_logits, dim=-1)

        loss = None
        if labels is not None:
            loss = F.cross_entropy(scaled_logits, labels, label_smoothing=label_smoothing)

        return {
            "logits": logits,
            "scaled_logits": scaled_logits,
            "probabilities": probs,
            "loss": loss,
        }

print("Arsitektur SemanticDecisionModel siap digunakan.")

Arsitektur SemanticDecisionModel siap digunakan.


## 4. Utilitas Pemrosesan Teks & Collate Batch Dinamis

In [4]:
from typing import Dict, Any, List
from torch.utils.data import Dataset

def format_decision_row(tokenizer, state: str, question: str, criteria: Dict[str, str], gold_option: str = None, max_len: int = 4096):
    mask_token = tokenizer.mask_token or "<mask"
    lines = [f"State: {state.strip()}", f"Question: {question.strip()}", "Options:"]

    option_keys = list(criteria.keys())
    if not option_keys:
        return None

    for opt_key, opt_desc in criteria.items():
        desc_clean = str(opt_desc).strip() if opt_desc else opt_key
        lines.append(f"{mask_token} {opt_key}: {desc_clean}")

    full_text = "\n".join(lines)
    enc = tokenizer(full_text, max_length=max_len, truncation=True, return_tensors="pt")
    input_ids = enc["input_ids"][0]
    attn_mask = enc["attention_mask"][0]

    mask_id = tokenizer.mask_token_id
    mask_indices = (input_ids == mask_id).nonzero(as_tuple=True)[0].tolist()

    # Jika opsi terpotong melebihi max_len, abaikan baris ini
    if len(mask_indices) != len(option_keys):
        return None

    label_idx = option_keys.index(gold_option) if (gold_option and gold_option in option_keys) else -1

    return {
        "input_ids": input_ids,
        "attention_mask": attn_mask,
        "mask_indices": mask_indices,
        "options": option_keys,
        "label": label_idx,
    }

def collate_fn(batch: List[Dict[str, Any]], pad_token_id: int = 1):
    max_seq_len = max(len(b["input_ids"]) for b in batch)
    max_opts = max(len(b["mask_indices"]) for b in batch)
    B = len(batch)

    input_ids = torch.full((B, max_seq_len), pad_token_id, dtype=torch.long)
    attention_mask = torch.zeros((B, max_seq_len), dtype=torch.long)
    mask_positions = torch.zeros((B, max_opts), dtype=torch.long)
    mask_valid = torch.zeros((B, max_opts), dtype=torch.bool)
    labels = torch.zeros(B, dtype=torch.long)

    for i, b in enumerate(batch):
        L = len(b["input_ids"])
        input_ids[i, :L] = b["input_ids"]
        attention_mask[i, :L] = b["attention_mask"]

        n_o = len(b["mask_indices"])
        mask_positions[i, :n_o] = torch.tensor(b["mask_indices"], dtype=torch.long)
        mask_valid[i, :n_o] = True
        labels[i] = b["label"]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "mask_positions": mask_positions,
        "mask_valid": mask_valid,
        "labels": labels,
    }

class ListDataset(Dataset):
    def __init__(self, data):
        self.data = data
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return self.data[idx]

## 5. Pemuatan & Penggabungan Data Mixture 3 Sumber

Menggabungkan:
1. **`LocalLLaMA/typed-decisions`**: 100% data train (1.200 baris)
2. **`legacy-datasets/banking77`**: 2.500 baris train (skema penuh 77 opsi)
3. **`clinc/clinc_oos`**: 2.500 baris train (skema penuh 150+oos opsi)

In [5]:
import json
import random
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel

print(f"Inisialisasi Tokenizer & Backbone: {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
backbone = AutoModel.from_pretrained(MODEL_ID)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SemanticDecisionModel(backbone, hidden_size=backbone.config.hidden_size).to(device)
print(f"Model Semantic Decision Engine aktif di: {device}")

rng = random.Random(SEED)
all_samples = []

# 1. Load Typed Decisions (Train)
print("\n[1/3] Memuat LocalLLaMA/typed-decisions (Train)...", flush=True)
ds_td = load_dataset("LocalLLaMA/typed-decisions", "all", split="train")
n_td = 0
for row in ds_td:
    try:
        qs = json.loads(row["questions"]) if isinstance(row["questions"], str) else row["questions"]
        gs = json.loads(row["gold"]) if isinstance(row["gold"], str) else row["gold"]
        for qid, qdata in qs.items():
            if qdata.get("type") == "choice" and "criteria" in qdata:
                g_entry = gs.get(qid)
                g_ans = (g_entry.get("label") or g_entry.get("choice") or g_entry.get("answer")) if isinstance(g_entry, dict) else g_entry
                if g_ans and str(g_ans) in qdata["criteria"]:
                    item = format_decision_row(
                        tokenizer,
                        state=row["state"],
                        question=qdata.get("instructions", "Select best option:"),
                        criteria=qdata["criteria"],
                        gold_option=str(g_ans),
                    )
                    if item and item["label"] != -1:
                        all_samples.append(item)
                        n_td += 1
                        break
    except Exception:
        continue
print(f"  -> Berhasil memuat {n_td} baris Typed Decisions.")

# 2. Load BANKING77 (Train)
print("\n[2/3] Memuat BANKING77 (Train)...", flush=True)
ds_b77 = load_dataset("legacy-datasets/banking77", split="train")
lbl_names = ds_b77.features["label"].names
n_b77 = 0
indices_b77 = list(range(len(ds_b77)))
rng.shuffle(indices_b77)
for idx in indices_b77:
    if n_b77 >= N_SAMPLES_BANKING:
        break
    row = ds_b77[idx]
    gold = lbl_names[row["label"]]
    crit = {k: k.replace("_", " ") for k in lbl_names}  # skema penuh 77 opsi
    item = format_decision_row(
        tokenizer,
        state=row["text"],
        question="Which banking intent does this message express?",
        criteria=crit,
        gold_option=gold,
    )
    if item and item["label"] != -1:
        all_samples.append(item)
        n_b77 += 1
print(f"  -> Berhasil memuat {n_b77} baris BANKING77.")

# 3. Load CLINC150 (Train)
print("\n[3/3] Memuat CLINC150 (Train)...", flush=True)
ds_clinc = load_dataset("clinc/clinc_oos", "plus", split="train")
int_names = ds_clinc.features["intent"].names
n_clinc = 0
indices_clinc = list(range(len(ds_clinc)))
rng.shuffle(indices_clinc)
for idx in indices_clinc:
    if n_clinc >= N_SAMPLES_CLINC:
        break
    row = ds_clinc[idx]
    gold = int_names[row["intent"]]
    crit = {k: k.replace("_", " ") for k in int_names}  # skema penuh 150+oos opsi
    item = format_decision_row(
        tokenizer,
        state=row["text"],
        question="What is the intent of this customer inquiry?",
        criteria=crit,
        gold_option=gold,
    )
    if item and item["label"] != -1:
        all_samples.append(item)
        n_clinc += 1
print(f"  -> Berhasil memuat {n_clinc} baris CLINC150.")

# Acak campuran data & buat 90/10 split
rng.shuffle(all_samples)
split_idx = int(0.9 * len(all_samples))
train_data = all_samples[:split_idx]
val_data = all_samples[split_idx:]

print(f"\nTotal Data Mixture Siap: {len(train_data)} train samples, {len(val_data)} validation samples.")
assert len(train_data) > 0, "Error: Data train kosong!"

Inisialisasi Tokenizer & Backbone: FacebookAI/xlm-roberta-base...


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

...

plus/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/15250 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3100 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5500 [00:00<?, ? examples/s]

  -> Berhasil memuat 2500 baris CLINC150.

Total Data Mixture Siap: 5526 train samples, 615 validation samples.


## 6. Training Loop dengan Differentiated Learning Rate

* Backbone XLM-RoBERTa: `lr = 2e-5`
* Semantic Projection Head: `lr = 5e-4`
* Scale & Temperature parameters: `lr = 1e-3`

In [6]:
import time
from tqdm.auto import tqdm
from torch.utils.data import DataLoader
from transformers import get_cosine_schedule_with_warmup

pad_id = tokenizer.pad_token_id or 1
train_loader = DataLoader(ListDataset(train_data), batch_size=BATCH_SIZE, shuffle=True, collate_fn=lambda b: collate_fn(b, pad_id))
val_loader = DataLoader(ListDataset(val_data), batch_size=BATCH_SIZE, shuffle=False, collate_fn=lambda b: collate_fn(b, pad_id))

# Differentiated Learning Rates
optimizer = torch.optim.AdamW([
    {"params": model.backbone.parameters(), "lr": LR_BACKBONE},
    {"params": model.state_proj.parameters(), "lr": LR_HEAD},
    {"params": model.opt_proj.parameters(), "lr": LR_HEAD},
    {"params": [model.scale, model.log_temp], "lr": 1e-3},
], weight_decay=0.01)

total_steps = len(train_loader) * EPOCHS
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=int(0.05 * total_steps), num_training_steps=total_steps)

print(f"=== Memulai Training: {EPOCHS} Epoch, {total_steps} Total Steps ===")
best_vacc, best_ep, best_state = -1.0, 0, None
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")
    optimizer.zero_grad()
    for step, batch in enumerate(pbar):
        optimizer.zero_grad()
        inp = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        pos = batch["mask_positions"].to(device)
        val = batch["mask_valid"].to(device)
        labels = batch["labels"].to(device)

        out = model(inp, mask, pos, val, labels=labels)
        raw_loss = out["loss"]
        (raw_loss / GRAD_ACCUM).backward()
        if (step + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += raw_loss.item()
        preds = out["probabilities"].argmax(dim=-1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        pbar.set_postfix({"loss": f"{loss.item():.4f}", "acc": f"{correct/total*100:.1f}%"})

    train_acc = correct / total
    print(f"Selesai Epoch {epoch}: Avg Loss = {total_loss/len(train_loader):.4f} | Acc = {train_acc*100:.2f}%")
    model.eval()
    v_correct, v_total = 0, 0
    with torch.no_grad():
        for vb in val_loader:
            vout = model(vb["input_ids"].to(device), vb["attention_mask"].to(device), vb["mask_positions"].to(device), vb["mask_valid"].to(device))
            vpred = vout["probabilities"].argmax(dim=-1)
            v_correct += (vpred == vb["labels"].to(device)).sum().item()
            v_total += vb["labels"].size(0)
    vacc = v_correct / max(v_total, 1)
    print(f"  val acc = {vacc*100:.2f}%", flush=True)
    if vacc > best_vacc:
        best_vacc, best_ep = vacc, epoch
        best_state = {k: v.cpu() for k, v in model.state_dict().items()}
model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
print(f"Checkpoint terbaik: epoch {best_ep} (val {best_vacc*100:.2f}%)")

=== Memulai Training: 5 Epoch, 1730 Total Steps ===


Epoch 1/5:   0%|          | 0/346 [00:00<?, ?it/s]

Selesai Epoch 1: Avg Loss = 54.5181 | Acc = 60.21%


Epoch 2/5:   0%|          | 0/346 [00:00<?, ?it/s]

Selesai Epoch 2: Avg Loss = 40.0771 | Acc = 82.25%


...

Selesai Epoch 3: Avg Loss = 31.8850 | Acc = 86.88%


Epoch 4/5:   0%|          | 0/346 [00:00<?, ?it/s]

Selesai Epoch 4: Avg Loss = 28.0822 | Acc = 89.03%


Epoch 5/5:   0%|          | 0/346 [00:00<?, ?it/s]

Selesai Epoch 5: Avg Loss = 26.8630 | Acc = 90.79%


## 7. Kalibrasi Suhu Pasca-Latih (*Post-Hoc L-BFGS Temperature Scaling*)

In [7]:
print("Mengoptimalkan suhu kalibrasi pada set validasi...")
model.eval()
all_row_logits, all_row_labels = [], []

with torch.no_grad():
    for batch in val_loader:
        inp = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        pos = batch["mask_positions"].to(device)
        val = batch["mask_valid"].to(device)
        out = model(inp, mask, pos, val)
        logits = out["logits"].cpu()
        valid = batch["mask_valid"].cpu()
        labels = batch["labels"].cpu()

        for b_i in range(logits.size(0)):
            n_opts = int(valid[b_i].sum().item())
            all_row_logits.append(logits[b_i, :n_opts])
            all_row_labels.append(int(labels[b_i].item()))

# Optimasi L-BFGS (mendukung opsi variabel per baris tanpa tensor cat mismatch)
temp_param = nn.Parameter(torch.ones(1))
opt_temp = torch.optim.LBFGS([temp_param], lr=0.02, max_iter=50)

def closure():
    opt_temp.zero_grad()
    T = temp_param.clamp(min=0.05, max=10.0)
    loss = sum(
        F.cross_entropy((z / T).unsqueeze(0), torch.tensor([g]))
        for z, g in zip(all_row_logits, all_row_labels)
    ) / max(len(all_row_logits), 1)
    loss.backward()
    return loss

opt_temp.step(closure)
calibrated_t = float(temp_param.detach().clamp(min=0.05, max=10.0).item())

with torch.no_grad():
    model.log_temp.copy_(torch.tensor(math.log(calibrated_t), device=model.log_temp.device))

print(f"Suhu Berhasil Dikalibrasi: T = {calibrated_t:.3f}")

Mengoptimalkan suhu kalibrasi pada set validasi...
Suhu Berhasil Dikalibrasi: T = 1.453


## 8. Evaluasi 3 Benchmark Standar (**Typed Decisions**, **BANKING77**, & **CLINC150**)

In [8]:
def evaluate_dataset(model, loader, device, n_bins=10, meta=None):
    model.eval()
    all_probs, all_preds, all_labels, all_full = [], [], [], []

    with torch.no_grad():
        for batch in loader:
            inp = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            pos = batch["mask_positions"].to(device)
            val = batch["mask_valid"].to(device)
            labels = batch["labels"]

            out = model(inp, mask, pos, val)
            probs = out["probabilities"].cpu()
            max_p, preds = probs.max(dim=-1)

            all_full.extend(probs.tolist())
            all_probs.extend(max_p.tolist())
            all_preds.extend(preds.tolist())
            all_labels.extend(labels.tolist())

    corrects = [int(p == l) for p, l in zip(all_preds, all_labels)]
    acc = sum(corrects) / max(len(corrects), 1)

    # Hitung Expected Calibration Error (ECE)
    bin_boundaries = [i / n_bins for i in range(n_bins + 1)]
    ece = 0.0
    total_samples = len(all_probs)

    for i in range(n_bins):
        bin_lower, bin_upper = bin_boundaries[i], bin_boundaries[i + 1]
        in_bin = [j for j, p in enumerate(all_probs) if bin_lower <= p < bin_upper]
        if in_bin:
            bin_acc = sum(corrects[j] for j in in_bin) / len(in_bin)
            bin_conf = sum(all_probs[j] for j in in_bin) / len(in_bin)
            ece += (len(in_bin) / total_samples) * abs(bin_acc - bin_conf)

    return {"accuracy": acc, "ece": ece, "total": total_samples}, to_rows(meta, all_full, all_preds)

def to_rows(meta, all_full, all_preds):
    rows = []
    if meta is None:
        return rows
    for (g, opts), fv, pr in zip(meta, all_full, all_preds):
        p = {str(o): float(v) for o, v in zip(opts, fv)}
        best = max(p, key=p.__getitem__)
        rows.append({"gold": g, "options": opts, "pred": best, "probs": p})
    return rows

bench_results, bench_dumps = {}, {}

# 1. Typed Decisions Test Set (400 sampel test resmi)
print("Evaluasi Typed Decisions Test Set...")
ds_td = load_dataset("LocalLLaMA/typed-decisions", "all", split="test")
items_td, meta_td = [], []
for row in ds_td:
    try:
        qs = json.loads(row["questions"]) if isinstance(row["questions"], str) else row["questions"]
        gs = json.loads(row["gold"]) if isinstance(row["gold"], str) else row["gold"]
        for qid, qdata in qs.items():
            if qdata.get("type") == "choice" and "criteria" in qdata:
                g_entry = gs.get(qid)
                g_ans = (g_entry.get("label") or g_entry.get("choice") or g_entry.get("answer")) if isinstance(g_entry, dict) else g_entry
                if g_ans and str(g_ans) in qdata["criteria"]:
                    f = format_decision_row(tokenizer, row["state"], qdata.get("instructions", "Option:"), qdata["criteria"], str(g_ans))
                    if f and f["label"] != -1:
                        items_td.append(f)
                        meta_td.append((str(g_ans), list(f["options"])))
                        break
    except Exception:
        continue
loader_td = DataLoader(ListDataset(items_td), batch_size=BATCH_SIZE, collate_fn=lambda b: collate_fn(b, pad_id))
bench_results["typed_decisions"], bench_dumps["typed_decisions"] = evaluate_dataset(model, loader_td, device, meta=meta_td)

# 2. BANKING77 (Test Set resmi penuh 3.080, skema penuh 77 opsi)
print("Evaluasi BANKING77 Test Set...")
ds_b77_test = load_dataset("legacy-datasets/banking77", split="test")
lbl_names = ds_b77_test.features["label"].names
items_b77, meta_b77 = [], []
for row in ds_b77_test:
    gold = lbl_names[row["label"]]
    crit = {k: k.replace("_", " ") for k in lbl_names}
    f = format_decision_row(tokenizer, row["text"], "Which banking intent?", crit, gold)
    if f and f["label"] != -1:
        items_b77.append(f)
        meta_b77.append((gold, list(f["options"])))
loader_b77 = DataLoader(ListDataset(items_b77), batch_size=BATCH_SIZE, collate_fn=lambda b: collate_fn(b, pad_id))
bench_results["banking77"], bench_dumps["banking77"] = evaluate_dataset(model, loader_b77, device, meta=meta_b77)

# 3. CLINC150 (Test Set resmi penuh 5.500 + OOS slice)
print("Evaluasi CLINC150 Test Set...")
ds_clinc_test = load_dataset("clinc/clinc_oos", "plus", split="test")
int_names = ds_clinc_test.features["intent"].names
items_clinc, meta_clinc = [], []
for row in ds_clinc_test:
    gold = int_names[row["intent"]]
    crit = {k: k.replace("_", " ") for k in int_names}
    f = format_decision_row(tokenizer, row["text"], "What is the customer's intent?", crit, gold)
    if f and f["label"] != -1:
        items_clinc.append(f)
        meta_clinc.append((gold, list(f["options"])))
loader_clinc = DataLoader(ListDataset(items_clinc), batch_size=BATCH_SIZE, collate_fn=lambda b: collate_fn(b, pad_id))
bench_results["clinc150"], bench_dumps["clinc150"] = evaluate_dataset(model, loader_clinc, device, meta=meta_clinc)

# Cetak Tabel Hasil Akhir
print("\n" + "="*65)
print(f"| {'Benchmark Dataset':18} | {'Akurasi':8} | {'ECE (Kalibrasi)':16} | {'Sampel':6} |")
print("| :--- | :---: | :---: | :---: |")
for bname, m in bench_results.items():
    print(f"| {bname:18} | {m['accuracy']*100:6.2f}% | {m['ece']:16.4f} | {m['total']:6} |")
print("="*65)
oos_rows = [r for r in bench_dumps["clinc150"] if "oos" in str(r["gold"]).lower()]
oos_rec = sum(1 for r in oos_rows if r["pred"] == r["gold"]) / max(len(oos_rows), 1)
print(f"OOS recall: {oos_rec*100:.2f}% (n={len(oos_rows)})")
items_pred = []
for bname, rows in bench_dumps.items():
    for r in rows:
        items_pred.append({"bench": bname, "gold": {"q": r["gold"]}, "answers": {"q": {"type": "choice", "choice": r["pred"], "confidence": r["probs"][r["pred"]], "probabilities": r["probs"], "answer": r["pred"]}}})
os.makedirs(OUTPUT_DIR, exist_ok=True)
with open(os.path.join(OUTPUT_DIR, "items_pred.json"), "w") as f:
    json.dump(items_pred, f)
with open(os.path.join(OUTPUT_DIR, "summary_eval.json"), "w") as f:
    json.dump({"bench_results": bench_results, "oos_recall": oos_rec, "seed": SEED}, f, indent=1, default=str)
print("Tersimpan di OUTPUT_DIR (ikut ter-upload ke HF): items_pred.json + summary_eval.json")

Evaluasi Typed Decisions Test Set...
Evaluasi BANKING77 Test Set...
Evaluasi CLINC150 Test Set...

| Benchmark Dataset  | Akurasi  | ECE (Kalibrasi)  | Sampel |
| :--- | :---: | :---: | :---: |
| typed_decisions    |  50.26% |           0.1414 |    382 |
| banking77          |  94.80% |           0.0366 |    500 |
| clinc150           |  95.40% |           0.0298 |    500 |


## 9. Simpan Model Lokal & Push ke Hugging Face Hub

In [12]:
import os

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Menyimpan model ke direktori: {OUTPUT_DIR}...")

tokenizer.save_pretrained(OUTPUT_DIR)
model.backbone.save_pretrained(OUTPUT_DIR)

head_info = {
    "state_proj": model.state_proj.state_dict(),
    "opt_proj": model.opt_proj.state_dict(),
    "scale": float(model.scale.detach().item()),
    "temperature": calibrated_t,
    "hidden_size": backbone.config.hidden_size,
    "model_id": MODEL_ID,
    "bench_results": bench_results,
}
torch.save(head_info, os.path.join(OUTPUT_DIR, "decision_head.pt"))
with open(os.path.join(OUTPUT_DIR, "decision_meta.json"), "w") as f:
    json.dump(head_info, f, indent=2, default=str)

print("File bobot model berhasil disimpan lokal!")

if HF_REPO and HF_TOKEN:
    from huggingface_hub import HfApi
    print(f"Mengunggah ke Hugging Face Hub: {HF_REPO}...")
    api = HfApi(token=HF_TOKEN)
    api.create_repo(repo_id=HF_REPO, exist_ok=True, private=False)
    api.upload_folder(
        folder_path=OUTPUT_DIR,
        repo_id=HF_REPO,
        commit_message="Upload XLM-RoBERTa System One Multilingual Decision Model with Semantic Head",
    )
    print(f"Model Sukses Diunggah: https://huggingface.co/{HF_REPO}")
else:
    print("HF_REPO atau HF_TOKEN tidak disetel. Unggah ke Hub dilewati.")
run_meta = {"model": MODEL_ID, "seed": SEED, "epochs": EPOCHS, "batch": BATCH_SIZE, "grad_accum": GRAD_ACCUM, "lr_backbone": LR_BACKBONE, "lr_head": LR_HEAD, "best_epoch": best_ep, "best_val": best_vacc, "temperature": calibrated_t, "bench_results": bench_results, "oos_recall": oos_rec}
with open(os.path.join(OUTPUT_DIR, "run_meta.json"), "w") as f:
    json.dump(run_meta, f, indent=2, default=str)
print("run_meta.json tersimpan (tempel ke docs/RUN_LOG.md)")

Menyimpan model ke direktori: xlm-roberta-systemone-decision...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

File bobot model berhasil disimpan lokal!
Mengunggah ke Hugging Face Hub: itsdevice/xlm-roberta-systemone...
Model Sukses Diunggah: https://huggingface.co/itsdevice/xlm-roberta-systemone


## 10. Pengujian Inferensi Multilingual $O(1)$ (Bahasa Indonesia & English)

Menguji apakah model sekarang mampu memilih opsi yang tepat dengan tingkat keyakinan (confidence) yang tinggi dan tegas.

In [10]:
def predict(model, tokenizer, state, question, criteria, device):
    model.eval()
    fmt = format_decision_row(tokenizer, state, question, criteria)
    batch = collate_fn([fmt], tokenizer.pad_token_id or 1)
    with torch.no_grad():
        out = model(
            input_ids=batch["input_ids"].to(device),
            attention_mask=batch["attention_mask"].to(device),
            mask_positions=batch["mask_positions"].to(device),
            mask_valid=batch["mask_valid"].to(device),
        )
        probs = out["probabilities"][0, :len(fmt["options"])].cpu().tolist()

    best_idx = int(torch.tensor(probs).argmax().item())
    return {
        "answer": fmt["options"][best_idx],
        "confidence": round(probs[best_idx], 4),
        "probabilities": {opt: round(p, 4) for opt, p in zip(fmt["options"], probs)}
    }

# 1. Kasus Bahasa Indonesia
res_id = predict(
    model, tokenizer,
    state="Kartu debit saya tertelan di mesin ATM dan saldo terpotong dua kali tanpa uang keluar.",
    question="Tim mana yang harus menangani tiket ini?",
    criteria={
        "billing_dispute": "Masalah sengketa tagihan, debit ganda, pengembalian dana saldo",
        "hardware_atm": "Kerusakan fisik mesin ATM, kartu tertelan, mesin error",
        "sales_marketing": "Pembukaan rekening baru, promosi kartu kredit, promo bunga"
    },
    device=device
)
print("=== Uji Kasus 1 (Bahasa Indonesia) ===")
print("Jawaban Terpilih:", res_id["answer"])
print("Confidence Terkalibrasi:", res_id["confidence"])
print("Distribusi Probabilitas:", res_id["probabilities"])

# 2. Kasus English (Engineering Routing)
res_en = predict(
    model, tokenizer,
    state="API returning 504 Gateway Timeout during peak hours on user login microservice.",
    question="Which engineering squad should be paged?",
    criteria={
        "infrastructure": "Server capacity, network gateways, container scaling",
        "database_admin": "Query locks, slow schema migrations, disk full",
        "frontend_ui": "CSS glitches, browser cache, mobile client layout"
    },
    device=device
)
print("\n=== Uji Kasus 2 (English) ===")
print("Jawaban Terpilih:", res_en["answer"])
print("Confidence Terkalibrasi:", res_en["confidence"])
print("Distribusi Probabilitas:", res_en["probabilities"])

# 3. Kasus Perbankan (Banking Intent Routing)
res_bank = predict(
    model, tokenizer,
    state="How long does it take for my new replacement Mastercard to arrive in the mail?",
    question="Which intent does this banking customer inquiry express?",
    criteria={
        "card_arrival": "Estimates and tracking for when a replacement card arrives",
        "change_pin": "Updating or resetting your card security pin number",
        "lost_stolen_card": "Reporting that a card was stolen or misplaced"
    },
    device=device
)
print("\n=== Uji Kasus 3 (Banking Intent) ===")
print("Jawaban Terpilih:", res_bank["answer"])
print("Confidence Terkalibrasi:", res_bank["confidence"])
print("Distribusi Probabilitas:", res_bank["probabilities"])

=== Uji Kasus 1 (Bahasa Indonesia) ===
Jawaban Terpilih: hardware_atm
Confidence Terkalibrasi: 0.9455
Distribusi Probabilitas: {'billing_dispute': 0.0378, 'hardware_atm': 0.9455, 'sales_marketing': 0.0167}

=== Uji Kasus 2 (English) ===
Jawaban Terpilih: infrastructure
Confidence Terkalibrasi: 0.9516
Distribusi Probabilitas: {'infrastructure': 0.9516, 'database_admin': 0.0252, 'frontend_ui': 0.0232}

=== Uji Kasus 3 (Banking Intent) ===
Jawaban Terpilih: card_arrival
Confidence Terkalibrasi: 0.9995
Distribusi Probabilitas: {'card_arrival': 0.9995, 'change_pin': 0.0003, 'lost_stolen_card': 0.0003}
